This file is part of the CRISPRsummerschool 2026 exercises

Copyright (c) 2023-26 Christian Anthon & 2026 Gül Sude Demircan

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, version 3.
# Convolutions in CRISPR on-target
Below you will find the third exercise.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RTH-tools/CRISPRsummerschool/blob/main/2026/CRISPR/exercise/crispr_2026_crispr_exercise3.ipynb)

In this exercise we will take a look at what are the actual outcome of the convolutions of the on-target sequence in the deep learning model.

In the code below, we will re-use the simplified version of the CRISPRon ontarget model we created in the previous exercise, however we have reduced the number of convolutions to 10.

## basic code definitions
Enter the cell below and press play or Ctrl+Enter in the block below to execute. You should see the message "Definitions executed" printed after execution.

In [ ]:
#!/usr/bin/env python3
# CRISPRsummerschool 2026 -- PyTorch version
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


eLENGTH30 = 30
eDEPTH = 4

# Function to onehot encode the data
def onehot(x):
    z = list()
    for y in list(x):
        if y in "Aa":
            z.append(0)
        elif y in "Cc":
            z.append(1)
        elif y in "Gg":
            z.append(2)
        elif y in "TtUu":
            z.append(3)
        else:
            print("Non-ATGCU character in", x)
            raise Exception
    return z

# Function to set the data into the appropriate format
def set_data(DX, s, mask=None):
    # mask should be a list of length len(s) of 1s and 0s: positions where mask
    # is 0 are onehot encoded, positions with 1 are masked out (left as zeros).
    if s is None:
        return
    assert(mask == None or (type(mask) is list and len(mask) == len(s)))
    if type(mask) is list:
        for j, x in enumerate(onehot(s)):
            if mask[j] == 0:
                DX[j][x] = 1
    else:
        for j, x in enumerate(onehot(s)):
            DX[j][x] = 1

# Preprocessing function for the sequence data
def preprocess_seq(data, mask=None, use_dgb=True):
    DATA_X30 = np.zeros((len(data), eLENGTH30, eDEPTH), dtype=np.float32)  # onehot
    DATA_G = np.zeros((len(data), 1), dtype=np.float32)  # deltaGb
    DATA_Y = np.zeros((len(data)), dtype=np.float32)  # efficiency

    for l, d in enumerate(data):
        set_data(DATA_X30[l], d[1], mask)
        if use_dgb:
            DATA_G[l] = -d[2]
        DATA_Y[l] = d[3]
    return (DATA_X30, DATA_G, DATA_Y)


# Convert numpy arrays to torch tensors on `device`.
# NOTE: the one-hot stays (N, 30, 4) here; the model permutes it to
#       (N, 4, 30) internally, because PyTorch Conv1d expects the layout
#       (batch, channels, length) -- this is the main Keras -> PyTorch change.
def to_tensors(x30, g, y):
    return (torch.from_numpy(x30).to(device),
            torch.from_numpy(g).to(device),
            torch.from_numpy(y).to(device))

def evaluate(model, data):
    """Return (mse, mae) on a (Xc, Xg, y) split. Runs in eval mode (dropout OFF)."""
    Xc, Xg, y = data
    model.eval()
    with torch.no_grad():
        pred = model(Xc, Xg).squeeze(-1)          # (N, 1) -> (N,)
        mse = torch.mean((pred - y) ** 2).item()
        mae = torch.mean(torch.abs(pred - y)).item()
    return mse, mae

def train(model, train_data, val_data, epochs=200, batch_size=64, lr=1e-3,
          patience=25, min_delta=0.1, verbose=True):
    """Mini-batch training with early stopping and restore-best-weights.

    Mirrors the original Keras callback
        EarlyStopping(monitor='val_loss', min_delta=0.1, patience=25,
                      restore_best_weights=True)
    """
    Xc, Xg, y = train_data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait, history = float("inf"), None, 0, []
    n = Xc.shape[0]
    for epoch in range(epochs):
        model.train()                             # dropout ON (Keras did this automatically)
        perm = torch.randperm(n, device=Xc.device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            pred = model(Xc[idx], Xg[idx]).squeeze(-1)   # (B,1) -> (B,): MUST squeeze
            loss = loss_fn(pred, y[idx])
            loss.backward()
            optimizer.step()
        val_mse, val_mae = evaluate(model, val_data)
        history.append(val_mse)
        if val_mse < best_val - min_delta:        # Keras "minimum improvement" rule
            best_val, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if verbose:
            print("epoch %3d  val_mse=%8.3f  val_mae=%6.3f  best=%8.3f  wait=%d"
                  % (epoch, val_mse, val_mae, best_val, wait))
        if wait >= patience:
            print("Early stopping at epoch %d (best val_mse=%.3f)" % (epoch, best_val))
            break
    if best_state is not None:
        model.load_state_dict(best_state)         # restore_best_weights=True
    return history


print("Definitions executed")

# ---- Robust data loader: identical behaviour in Colab and local Jupyter ----
# A file already in this folder is used as-is; a missing file is downloaded and
# its contents are validated. Pure Python (no shell), so it behaves the same in
# Colab, local Jupyter, Windows/Mac/Linux.
import urllib.request

DATA_SOURCES = {
    "training_data.csv": [
        "https://rth.dk/internal/index.php/s/S4jQMaER6nYAJGe/download",
    ],
    "validation_data.csv": [
        "https://rth.dk/internal/index.php/s/oHspJCgniRMog6r/download",
    ],
}

def _is_valid_csv(path):
    """A real data file starts with the known header, not an HTML error page."""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            first = fh.readline()
        return ("target" in first) and ("deltaGb" in first)
    except OSError:
        return False

def fetch_data(fname, dest_dir="."):
    """Return the path to a valid `fname`, downloading it only if needed.
    urllib raises on HTTP errors (unlike a bare `curl -o`) and we re-check the
    content, so a bad/expired URL fails loudly instead of silently writing an
    HTML page into a .csv."""
    dest = os.path.join(dest_dir, fname)
    if _is_valid_csv(dest):
        print("using existing", dest)
        return dest
    problems = []
    for url in DATA_SOURCES[fname]:
        try:
            print("downloading %s from %s ..." % (fname, url.split("/")[2]))
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            problems.append("%s -> %s" % (url, e))
            continue
        if _is_valid_csv(dest):
            return dest
        problems.append("%s -> downloaded file is not a valid CSV (an error page?)" % url)
    if os.path.exists(dest):
        os.remove(dest)                       # never leave a corrupt .csv behind
    raise RuntimeError(
        "Could not obtain %s. Upload it into this folder manually, or fix the "
        "URLs in DATA_SOURCES above.\nTried:\n  %s" % (fname, "\n  ".join(problems)))

fetch_data("training_data.csv")
fetch_data("validation_data.csv")


# Training / validation data
PATH = './'
d = pd.read_csv(PATH + 'training_data.csv').values.tolist()
dv = pd.read_csv(PATH + 'validation_data.csv').values.tolist()
(x30, g, y) = preprocess_seq(d)
(x30v, gv, yv) = preprocess_seq(dv)
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)
print('Data loaded')


DROPOUT_DENSE = 0.3
CONV_1_SIZE = 3
N_CONV_1 = 10
N_DENSE = 40
N_OUT = 40

class SimpleCRISPRon(nn.Module):
    """A simplified version of the CRISPRon on-target model (PyTorch).

    One 1D convolution over the one-hot sequence, followed by fully connected
    (dense) layers. The binding energy dGb is concatenated in *after* the first
    dense layer, exactly as in the original Keras model.
    """
    def __init__(self, n_conv=N_CONV_1, kernel=CONV_1_SIZE, n_dense=N_DENSE,
                 n_out=N_OUT, dropout=DROPOUT_DENSE, seq_len=eLENGTH30, depth=eDEPTH):
        super().__init__()
        self.conv = nn.Conv1d(depth, n_conv, kernel)      # (B,4,30) -> (B,n_conv,28)
        conv_out_len = seq_len - kernel + 1               # 30 - 3 + 1 = 28
        flat = n_conv * conv_out_len                      # flattened conv features
        self.collect = nn.Linear(flat, n_dense)           # "dense_0"
        self.dense1 = nn.Linear(n_dense + 1, n_dense)     # "dense_1"  (+1 = raw dGb)
        self.dense2 = nn.Linear(n_dense, n_out)           # "dense_2"
        self.dense3 = nn.Linear(n_out, n_out)             # "dense_on_off"
        self.out = nn.Linear(n_out, 1)                    # output (linear, no activation)
        self.drop = nn.Dropout(dropout)
        self.apply(self._xavier_uniform_init)

    def features(self, x, g):
        # Conv1d wants (batch, channels, length); our one-hot is
        # (batch, length=30, channels=4), so swap the last two axes.
        x = x.permute(0, 2, 1)                            # (B,30,4) -> (B,4,30)
        z = torch.relu(self.conv(x))                      # convolution + ReLU
        z = torch.flatten(z, start_dim=1)                 # (B, n_conv*28)
        z = self.drop(torch.relu(self.collect(z)))        # dense_0 + ReLU + dropout
        z = torch.cat([g, z], dim=1)                      # concat raw dGb (Keras order)
        z = self.drop(torch.relu(self.dense1(z)))         # dense_1
        z = self.drop(torch.relu(self.dense2(z)))         # dense_2
        z = self.drop(torch.relu(self.dense3(z)))         # dense_on_off
        return z

    def forward(self, x, g):
        return self.out(self.features(x, g))              # (B, 1)

    @staticmethod
    def _xavier_uniform_init(m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

model = SimpleCRISPRon().to(device)
print(model)
print('Model defined')


## Exercise 3.1 Model definition and convolutions
Look at the model printout (`print(model)`) above. You can also run a forward pass through just the convolution on one batch to see the output shape (see Exercise 3.4).

What are the dimensions (shape) of the output of the convolutional layer? (Remember that PyTorch uses the `(batch, channels, length)` ordering.)

How does the output shape relate to the input shape, the size of the convolutions, and to the number of convolutions? (*Hint: patching*)


In [ ]:
#answer

## Exercise 3.2 Model training

Execute the model **training** to initialize weights for later use. Check that you get more or less the same result as in exercise 1.

In [ ]:
print("training...")
LEARN = 1e-3
EPOCHS = 200
BATCH_SIZE = 64

# create a fresh (untrained) model and train it
model = SimpleCRISPRon().to(device)

history = train(model, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
                patience=25, min_delta=0.1)

print("done")
print("evaluating validation data using best model weights")
print(evaluate(model, (x30v_t, gv_t, yv_t)))


## Exercise 3.3
In PyTorch you access the convolutional layer directly as `model.conv`, and its kernel weights as the tensor `model.conv.weight`. Print them with `model.conv.weight.detach().cpu().numpy()`.

The convolution also has a separate bias tensor, `model.conv.bias`, which we can ignore for now.

What is the shape of `model.conv.weight`? What is the shape of the first of the 10 convolutions, `model.conv.weight[0]`?

(*Hint: in PyTorch a Conv1d weight has shape `(out_channels, in_channels, kernel_size)`, so each of the 10 filters is already laid out as `(4, 3)` - no transpose needed, unlike in Keras.*)


In [ ]:
#answer

## Exercise 3.4
Below we take the one-hot sequence of the first guide and put it into the form expected by a PyTorch `Conv1d` layer. Two things change compared to a single `(30, 4)` array:

1. we add a batch dimension (a `Conv1d` layer always works on a *batch* of inputs), and
2. we move the 4 nucleotide channels to the front, because `Conv1d` expects `(batch, channels, length)`.

So `(30, 4)` becomes `(1, 4, 30)`.


In [ ]:
print(x30[0].shape)                        # (30, 4): one-hot for the first guide
x30_0 = torch.from_numpy(x30[0]).to(device)
x30_0 = x30_0.permute(1, 0).unsqueeze(0)   # (30,4) -> (4,30) -> (1, 4, 30)
print(x30_0.shape)                         # (1, 4, 30): a batch of one, ready for Conv1d


### Exercise 3.4.1
Apply the convolution on the first ontarget sequence (`x30_0`) and examine the result. In the model the convolution is followed by a ReLU, so apply `torch.relu(model.conv(x30_0))`.

What are the output dimensions (shape)? (Recall the `(batch, channels, length)` ordering.)

Which part of the output comes from the first of the 10 convolutions?

In the output you will see only non-negative numbers and a lot of zeroes. Why is that, when the convolution weights can be both positive and negative?


In [ ]:
#answer

Below we define a very simple convolutional layer with just one convolution of size 3 without biases

In [ ]:
# One size-3 convolution over the 4 nucleotide channels.
# Original Keras weight shape was (kernel=3, in_channels=4, out_filters=1);
# a PyTorch Conv1d weight has shape (out_filters=1, in_channels=4, kernel=3).
w_keras = np.array([
    [[0.5], [0.0], [0.0], [0.0]],   # kernel position 0:  A -> 0.5
    [[0.5], [0.0], [0.0], [0.0]],   # kernel position 1:  A -> 0.5
    [[0.0], [0.0], [0.0], [0.0]],   # kernel position 2:  (all zero)
], dtype=np.float32)                # shape (3, 4, 1)
print("Keras-style weight, transposed:")
print(w_keras.transpose())

# Convert (kernel, in, out) -> (out, in, kernel) with permute(2, 1, 0)
w_torch = torch.from_numpy(w_keras).permute(2, 1, 0).contiguous()   # (1, 4, 3)
print("PyTorch conv weight shape:", tuple(w_torch.shape))

# define a simple convolutional layer with just one convolution of size 3 without bias
conv_simple = nn.Conv1d(eDEPTH, 1, CONV_1_SIZE, bias=False).to(device)
with torch.no_grad():
    conv_simple.weight.copy_(w_torch)


### Exercise 3.4.2
Apply the simple convolution `conv_simple` defined above on the first ontarget (`x30_0`) and relate the result to the input sequence of the first ontarget. (This filter has no ReLU, so you see its raw response - apply it directly as `conv_simple(x30_0)`.)

Convolutions are sometimes called filters. How does your observation of the result of the convolution match up with that?


In [ ]:
#answer

## Exercise 3.5 - Discussion
Convolutions are just one way of finding patterns in the input. Other architectures model sequences differently - recurrent networks (LSTMs) process a sequence step by step, and transformer / attention models (as used in language models such as BERT) let every position attend to every other position.

- Would a sequence model like an LSTM or a transformer be well suited to CRISPR **on-target** efficiency prediction? Why or why not?
- What about CRISPR **off-target** / specificity prediction, where a guide sequence has to be compared against many candidate target sites?


In [ ]:
#answer


---
**Continue to Exercise 4 - _CRISPRon with uncertainty estimation_**, where the model not only predicts efficiency but also reports *how confident* it is in each prediction.


## Reference

These exercises use a **simplified** version of the CRISPRon on-target efficiency model:

- Xiang, X., Corsi, G. I., Anthon, C., *et al.* (2021). Enhancing CRISPR-Cas9 gRNA efficiency prediction by data integration and deep learning. *Nature Communications*, **12**, 3238. https://doi.org/10.1038/s41467-021-23576-0
